# ISO/SAE 21434: Automotive Cybersecurity Engineering

**Module 05 · Cybersecurity (ISO/SAE 21434)** · Notebook 1 of 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/05_Cybersecurity_ISO21434/01_iso21434_fundamentals.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Beginner | ~50 min | [HARA (Module 03)](../03_Functional_Safety_ISO26262/02_hara.ipynb) recommended | ISO/SAE 21434:2021, UN R155, UN R156 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Explain the scope and lifecycle of ISO/SAE 21434:2021 and name its clause structure (clauses 5–15).
- List the seven TARA steps of clause 15 in the correct order.
- Apply the informative rating tables: impact (Severe/Major/Moderate/Negligible), attack feasibility (attack-potential and attack-vector based), the example risk matrix (risk value 1–5) and the example CAL table.
- Explain why a CAL is **not** a risk measure and **not** an ASIL equivalent.
- Place ISO/SAE 21434 next to UN R155/R156, ISO/SAE PAS 5112, ISO 24089 and the EU Cyber Resilience Act.

In [ ]:
# Setup: only standard scientific Python is needed (all in the Colab default image)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.ticker import MaxNLocator
from IPython.display import display

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': False})
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
rng = np.random.default_rng(42)
print("Setup complete.")

## 1. ISO/SAE 21434 overview

**ISO/SAE 21434:2021** *Road vehicles — Cybersecurity engineering* was published in August 2021 by ISO and SAE International.
It specifies requirements for cybersecurity risk management of **road-vehicle E/E systems, their components and interfaces**
across the whole lifecycle: concept, development, production, operation, maintenance and decommissioning.
It does not prescribe specific technologies or cryptographic algorithms; it prescribes *processes and work products*.

**Revision status (as of October 2026).** The 2021 first edition is current. A second edition is in early development:
presentations by the project leaders state that scope and structure stay unchanged, that ISO/SAE TR 8477 (verification and validation)
is to be merged in, and give publication dates around 2029, which are provisional
([GlobalPlatform, Dec 2025](https://globalplatform.org/wp-content/uploads/2025/12/7_ISO-SAE-2134-2nd-edition-Eaton_Krzeszewski.pdf);
[Omnex, Jul 2026](https://blog.omnex.com/2026/07/02/automotive-standards-update-spice-pam-4-0-iso-26262-ed-3-functional-safety-iso-21434-2/)).
Detailed guidance on CAL and *targeted attack feasibility* is being developed separately as ISO/SAE PAS 8475
(in the approval stages, not yet listed as published, as of October 2026; [ISO project page](https://www.iso.org/standard/83187.html)).

### Why cybersecurity matters for automated vehicles

Modern vehicles contain on the order of 100 ECUs, many external interfaces (cellular, Wi-Fi, Bluetooth, V2X, USB, diagnostics) and receive over-the-air (OTA) software updates.
For automated driving this matters because:

1. **Safety impact** — a successful attack can violate a safety goal (e.g. unintended emergency braking).
2. **Privacy** — cameras and location traces record identifiable people.
3. **Autonomy** — automated decisions rely on the integrity of sensor data, maps and V2X messages.
4. **Scalability** — one remote vulnerability can affect an entire fleet.

### Key principles

1. **Security by design** — built in from the concept phase, not added later.
2. **Defence in depth** — several independent layers of protection.
3. **Risk-based** — effort follows the TARA, not intuition.
4. **Lifecycle coverage** — monitoring and vulnerability management continue after start of production.
5. **Distributed activities** — responsibilities with suppliers are agreed in a cybersecurity interface agreement.

### Clause structure

| Clause | Title (abridged) | Typical work products |
|---|---|---|
| 5 | Organisational cybersecurity management | Cybersecurity policy, rules and processes, competence, quality and tool management |
| 6 | Project-dependent cybersecurity management | Cybersecurity plan, cybersecurity case, cybersecurity assessment, release for post-development |
| 7 | Distributed cybersecurity activities | Supplier capability, cybersecurity interface agreement (template in Annex C) |
| 8 | Continual cybersecurity activities | Cybersecurity monitoring, event evaluation, vulnerability analysis, vulnerability management |
| 9 | Concept | Item definition, cybersecurity goals and claims, cybersecurity concept |
| 10 | Product development | Cybersecurity specifications, integration and **verification** |
| 11 | Cybersecurity validation | Validation of cybersecurity goals at vehicle level (e.g. penetration testing) |
| 12 | Production | Production control plan |
| 13 | Operations and maintenance | Incident response, updates |
| 14 | End of cybersecurity support and decommissioning | Communication of end of support, decommissioning procedures |
| 15 | Threat analysis and risk assessment (TARA) methods | Methods used by clauses 8, 9 and 10 (Annexes E–H give informative examples) |

Clauses 1–4 cover scope, normative references, terms and general considerations. Annexes A–H are informative;
the rating tables used below come from Annexes E (CAL), F (impact rating), G (attack feasibility rating) and H (TARA example including a risk matrix).

In [ ]:
def visualize_cybersecurity_lifecycle():
    """Draw the ISO/SAE 21434 lifecycle with the supporting clauses."""
    fig, ax = plt.subplots(figsize=(15, 7))
    names = ['Concept\n(cl. 9)', 'Product\ndevelopment\n(cl. 10)', 'Validation\n(cl. 11)', 'Production\n(cl. 12)',
             'Operations &\nmaintenance\n(cl. 13)', 'End of support &\ndecommissioning\n(cl. 14)']
    phases = list(zip(names, np.linspace(0.08, 0.92, len(names))))
    for i, (name, x) in enumerate(phases):
        ax.add_patch(FancyBboxPatch((x - 0.06, 0.62), 0.12, 0.2, boxstyle='round,pad=0.01',
                                    facecolor=OKABE_ITO[4], edgecolor='black', alpha=0.85))
        ax.text(x, 0.72, name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        if i < len(phases) - 1:
            ax.annotate('', xy=(phases[i + 1][1] - 0.068, 0.72), xytext=(x + 0.068, 0.72),
                        arrowprops=dict(arrowstyle='->', lw=2))
    support = [('Organisational management (cl. 5)', 0.14), ('Project-dependent management (cl. 6)', 0.38),
               ('Distributed activities (cl. 7)', 0.62), ('Continual activities (cl. 8)\nmonitoring, vulnerability mgmt', 0.86)]
    for name, x in support:
        ax.add_patch(FancyBboxPatch((x - 0.1, 0.25), 0.2, 0.14, boxstyle='round,pad=0.01',
                                    facecolor=OKABE_ITO[1], edgecolor='black', alpha=0.6))
        ax.text(x, 0.32, name, ha='center', va='center', fontsize=8.5)
    ax.add_patch(FancyBboxPatch((0.02, 0.05), 0.96, 0.1, boxstyle='round,pad=0.01',
                                facecolor=OKABE_ITO[0], edgecolor='black', alpha=0.6))
    ax.text(0.5, 0.10, 'TARA methods (cl. 15) — used in concept (cl. 9), development (cl. 10) and continual activities (cl. 8)',
            ha='center', va='center', fontsize=9.5, fontweight='bold')
    ax.set_xlim(0, 1); ax.set_ylim(0, 0.95); ax.axis('off')
    ax.set_title('ISO/SAE 21434:2021 lifecycle (clauses 9–14) and supporting clauses (5–8, 15)', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()
    print('--- Lifecycle (text summary) ---')
    print('  Phases: ' + ' -> '.join(p[0].replace(chr(10), ' ') for p in phases))
    print('  Supporting: ' + '; '.join(s[0].replace(chr(10), ' ') for s in support))

visualize_cybersecurity_lifecycle()

## 2. TARA: Threat Analysis and Risk Assessment

TARA is defined as a set of methods in **clause 15**. The steps, in the order of the standard:

| Step | Sub-clause | Question | Output |
|---|---|---|---|
| 1 | 15.3 Asset identification | What has a cybersecurity property (C, I, A, authenticity, …) whose compromise leads to a **damage scenario**? | Assets with properties; damage scenarios |
| 2 | 15.4 Threat scenario identification | What could compromise that property, and how (asset + property + cause)? | Threat scenarios |
| 3 | 15.5 Impact rating | How bad is the damage scenario for the **road user**, in Safety, Financial, Operational and Privacy (SFOP) categories? | Severe / Major / Moderate / Negligible |
| 4 | 15.6 Attack path analysis | Which sequences of actions realise the threat scenario? | Attack paths (often as attack trees) |
| 5 | 15.7 Attack feasibility rating | How easy is each attack path? | High / Medium / Low / Very Low |
| 6 | 15.8 Risk value determination | Combine impact and feasibility | Risk value 1 (lowest) – 5 (highest) |
| 7 | 15.9 Risk treatment decision | Avoid, reduce, share or retain? | Decision per threat scenario → cybersecurity goals (reduce) or claims (share/retain) |

Notebook 2 of this module performs all seven steps end-to-end. Here we introduce the rating tables.

### Impact rating (Annex F)

Impact is rated per damage scenario for the **road user** (driver, passengers, pedestrians, other traffic participants).
OEM-specific losses (brand, warranty) may be rated as additional, organisation-defined categories.

| Rating | Safety (aligned with ISO 26262 S-classes) | Financial | Operational | Privacy |
|---|---|---|---|---|
| **Severe** | S3: life-threatening or fatal injuries | Catastrophic consequences the road user might not overcome | Loss or impairment of a core vehicle function | Significant harm, highly sensitive and easily linked data |
| **Major** | S2: severe injuries, survival probable | Substantial consequences the road user can overcome | Loss or impairment of an important vehicle function | Serious harm to the road user |
| **Moderate** | S1: light and moderate injuries | Inconvenient consequences the road user can overcome with limited resources | Partial degradation of a vehicle function | Moderate harm |
| **Negligible** | S0: no injuries | No effect, negligible consequences | No impairment perceived | No or negligible harm |

### Attack feasibility (Annex G)

ISO/SAE 21434 gives three example approaches. The **attack-potential approach** (G.2, values from ISO/IEC 18045) sums five factors;
a *higher* attack potential means the attack is *harder*, i.e. *lower* feasibility:

| Factor | Values |
|---|---|
| Elapsed time | ≤1 day **0** · ≤1 week **1** · ≤1 month **4** · ≤6 months **17** · >6 months **19** |
| Specialist expertise | Layman **0** · Proficient **3** · Expert **6** · Multiple experts **8** |
| Knowledge of the item | Public **0** · Restricted **3** · Confidential **7** · Strictly confidential **11** |
| Window of opportunity | Unlimited **0** · Easy **1** · Moderate **4** · Difficult **10** |
| Equipment | Standard **0** · Specialised **4** · Bespoke **7** · Multiple bespoke **9** |

| Attack potential (sum) | 0–13 | 14–19 | 20–24 | ≥25 |
|---|---|---|---|---|
| **Attack feasibility** | **High** | **Medium** | **Low** | **Very Low** |

There is no "Very High" feasibility class. The **CVSS-based approach** (G.3) maps the CVSS exploitability sub-score
(see Notebook 3), and the **attack-vector-based approach** (G.4) uses only the vector:
Network → High, Adjacent → Medium, Local → Low, Physical → Very Low.

### Risk value (example matrix, Annex H) and CAL (Annex E)

| Impact \ Feasibility | Very Low | Low | Medium | High |
|---|---|---|---|---|
| **Severe** | 2 | 3 | 4 | 5 |
| **Major** | 1 | 2 | 3 | 4 |
| **Moderate** | 1 | 2 | 2 | 3 |
| **Negligible** | 1 | 1 | 1 | 1 |

A **cybersecurity assurance level (CAL 1–4)** can be assigned to a *cybersecurity goal* to scale the rigour of the
assurance activities (e.g. depth of testing, independence of reviews). The informative example in Annex E uses the
**impact** and the **attack vector** — *not* the risk value and *not* the feasibility:

| Impact \ Attack vector | Physical | Local | Adjacent | Network |
|---|---|---|---|---|
| **Severe** | CAL 2 | CAL 3 | CAL 4 | CAL 4 |
| **Major** | CAL 1 | CAL 2 | CAL 3 | CAL 4 |
| **Moderate** | CAL 1 | CAL 1 | CAL 2 | CAL 3 |
| **Negligible** | — | — | — | — |

All of these tables are **informative examples**; an organisation may define its own methods and must document them.

In [ ]:
# ISO/SAE 21434:2021 informative tables (Annexes E, G, H). The same functions are available
# in the repository toolkit as avsafety.iso21434 (see repo avsafety/); results are identical.
IMPACT_LEVELS = ['Negligible', 'Moderate', 'Major', 'Severe']
FEASIBILITY_LEVELS = ['Very Low', 'Low', 'Medium', 'High']

ELAPSED_TIME = {'<=1 day': 0, '<=1 week': 1, '<=1 month': 4, '<=6 months': 17, '>6 months': 19}
EXPERTISE = {'layman': 0, 'proficient': 3, 'expert': 6, 'multiple experts': 8}
KNOWLEDGE = {'public': 0, 'restricted': 3, 'confidential': 7, 'strictly confidential': 11}
WINDOW = {'unlimited': 0, 'easy': 1, 'moderate': 4, 'difficult': 10}
EQUIPMENT = {'standard': 0, 'specialized': 4, 'bespoke': 7, 'multiple bespoke': 9}

def attack_potential(elapsed_time, expertise, knowledge, window, equipment):
    return (ELAPSED_TIME[elapsed_time] + EXPERTISE[expertise] + KNOWLEDGE[knowledge]
            + WINDOW[window] + EQUIPMENT[equipment])

def feasibility_from_potential(potential):
    if potential <= 13: return 'High'
    if potential <= 19: return 'Medium'
    if potential <= 24: return 'Low'
    return 'Very Low'

FEASIBILITY_FROM_VECTOR = {'Network': 'High', 'Adjacent': 'Medium', 'Local': 'Low', 'Physical': 'Very Low'}

RISK_MATRIX = {
    'Severe':     {'Very Low': 2, 'Low': 3, 'Medium': 4, 'High': 5},
    'Major':      {'Very Low': 1, 'Low': 2, 'Medium': 3, 'High': 4},
    'Moderate':   {'Very Low': 1, 'Low': 2, 'Medium': 2, 'High': 3},
    'Negligible': {'Very Low': 1, 'Low': 1, 'Medium': 1, 'High': 1},
}

CAL_TABLE = {
    'Severe':   {'Physical': 2, 'Local': 3, 'Adjacent': 4, 'Network': 4},
    'Major':    {'Physical': 1, 'Local': 2, 'Adjacent': 3, 'Network': 4},
    'Moderate': {'Physical': 1, 'Local': 1, 'Adjacent': 2, 'Network': 3},
}

def determine_cal(impact, attack_vector):
    """CAL for a cybersecurity goal (Annex E example); None for negligible impact."""
    return None if impact == 'Negligible' else CAL_TABLE[impact][attack_vector]

# Self-checks against the toolkit's unit tests
assert feasibility_from_potential(attack_potential('<=1 week', 'layman', 'public', 'unlimited', 'standard')) == 'High'
assert [feasibility_from_potential(x) for x in (13, 14, 19, 20, 24, 25)] == ['High', 'Medium', 'Medium', 'Low', 'Low', 'Very Low']
assert RISK_MATRIX['Severe']['High'] == 5 and determine_cal('Severe', 'Physical') == 2
print('Rating tables loaded; self-checks passed.')

### A first TARA example: pedestrian detection and AEB

The threat scenarios below were in the original version of this notebook; they are now rated with the standard's tables.
Each impact is rated from the road user's perspective (0 = Negligible … 3 = Severe), and the attack-potential factors are given explicitly so that the rating can be reviewed.
The full, traceable TARA (damage scenarios, attack paths, treatment and goals) follows in Notebook 2.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
threats = pd.DataFrame([
    # id, asset (property), threat scenario, attack vector, S, F, O, P, (elapsed, expertise, knowledge, window, equipment)
    ('T-001', 'Object list (integrity)', 'Physical adversarial patch suppresses pedestrian detection', 'Physical',
     3, 1, 1, 0, ('<=1 month', 'expert', 'restricted', 'easy', 'standard')),
    ('T-002', 'DNN weights (integrity)', 'Model tampering via a compromised OTA update', 'Network',
     3, 2, 2, 0, ('<=6 months', 'multiple experts', 'confidential', 'moderate', 'bespoke')),
    ('T-003', 'V2X messages (integrity)', 'Injection of a fake V2X pedestrian warning with a stolen certificate', 'Adjacent',
     2, 1, 1, 0, ('<=1 month', 'expert', 'restricted', 'easy', 'specialized')),
    ('T-004', 'ECU firmware (integrity)', 'Firmware tampering via the diagnostic (OBD-II) port', 'Local',
     3, 2, 2, 0, ('<=1 month', 'expert', 'confidential', 'moderate', 'specialized')),
    ('T-005', 'Camera stream (availability)', 'Camera dazzling with a high-power light source', 'Physical',
     2, 0, 1, 0, ('<=1 day', 'proficient', 'public', 'moderate', 'standard')),
    ('T-006', 'OTA channel (integrity)', 'Man-in-the-middle on the OTA download channel', 'Network',
     3, 2, 2, 1, ('<=6 months', 'expert', 'strictly confidential', 'easy', 'specialized')),
], columns=['id', 'asset', 'scenario', 'vector', 'S', 'F', 'O', 'P', 'factors'])

def rate(row):
    impact = IMPACT_LEVELS[max(row.S, row.F, row.O, row.P)]
    ap = attack_potential(*row.factors)
    feas = feasibility_from_potential(ap)
    return pd.Series({'impact': impact, 'attack_potential': ap, 'feasibility': feas,
                      'feasibility_vector_method': FEASIBILITY_FROM_VECTOR[row.vector],
                      'risk': RISK_MATRIX[impact][feas], 'CAL': determine_cal(impact, row.vector)})

df_tara = pd.concat([threats, threats.apply(rate, axis=1)], axis=1)
display(df_tara[['id', 'scenario', 'vector', 'impact', 'attack_potential', 'feasibility',
                 'feasibility_vector_method', 'risk', 'CAL']])

print('--- TARA (text summary) ---')
for _, r in df_tara.iterrows():
    print(f"  {r.id}: impact {r.impact}, attack potential {r.attack_potential} -> {r.feasibility} feasibility, "
          f"risk {r.risk}, CAL {r.CAL} ({r.vector})")
disagree = df_tara[df_tara.feasibility != df_tara.feasibility_vector_method]
print(f"\nAttack-potential and attack-vector methods disagree for {len(disagree)} of {len(df_tara)} threats: "
      + ', '.join(f"{r.id} ({r.feasibility} vs {r.feasibility_vector_method})" for _, r in disagree.iterrows()))
hi_risk_low_cal = df_tara[(df_tara.risk >= 4) & (df_tara.CAL <= 2)]
print('Threats with risk >= 4 but CAL <= 2 (CAL is not a risk measure): '
      + (', '.join(hi_risk_low_cal.id) if len(hi_risk_low_cal) else 'none'))

**Reading the result.**
- The attack-vector method is quick but coarse. It rates every "Physical" attack as Very Low feasibility. Physical-world attacks on *sensors* (patches, dazzling) need no access to the vehicle, so the attack-potential method rates them much higher. Use the vector method only for early, rough screening.
- A fleet-wide impact (T-002, T-006) does not make the risk high on its own; the risk value combines impact **and** feasibility.
- CAL and risk answer different questions. CAL scales the rigour of assurance for a goal, based on impact and how remotely the item can be reached. Risk drives the treatment decision.

In [ ]:
def visualize_tara_results(df):
    """Risk distribution, CAL distribution, impact profile and risk matrix for the example TARA."""
    df = df.copy()  # no side effects on the caller's DataFrame
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))

    risk_counts = df['risk'].value_counts().reindex([1, 2, 3, 4, 5], fill_value=0)
    ax1.bar(risk_counts.index, risk_counts.values, color=plt.cm.viridis(np.linspace(0.15, 0.9, 5)), edgecolor='black')
    ax1.set_title('Threat scenarios per risk value', fontweight='bold')
    ax1.set_xlabel('Risk value (1 = lowest, 5 = highest)'); ax1.set_ylabel('Number of threat scenarios')
    ax1.set_xticks([1, 2, 3, 4, 5]); ax1.yaxis.set_major_locator(MaxNLocator(integer=True))

    cal_counts = df['CAL'].fillna(0).astype(int).value_counts().reindex([0, 1, 2, 3, 4], fill_value=0)
    ax2.bar(['none', 'CAL 1', 'CAL 2', 'CAL 3', 'CAL 4'], cal_counts.values, color=OKABE_ITO[4], edgecolor='black')
    ax2.set_title('CAL per (candidate) cybersecurity goal', fontweight='bold')
    ax2.set_xlabel('Cybersecurity assurance level'); ax2.set_ylabel('Number of goals')
    ax2.yaxis.set_major_locator(MaxNLocator(integer=True))

    impact_data = df[['S', 'F', 'O', 'P']].T
    impact_data.columns = df['id']
    im = ax3.imshow(impact_data.values, cmap='viridis', vmin=0, vmax=3, aspect='auto')
    ax3.set_xticks(range(len(df))); ax3.set_xticklabels(df['id'])
    ax3.set_yticks(range(4)); ax3.set_yticklabels(['Safety', 'Financial', 'Operational', 'Privacy'])
    for i in range(4):
        for j in range(len(df)):
            v = impact_data.values[i, j]
            ax3.text(j, i, IMPACT_LEVELS[v], ha='center', va='center', fontsize=8, color='white' if v < 2 else 'black')
    cb = fig.colorbar(im, ax=ax3, ticks=[0, 1, 2, 3]); cb.ax.set_yticklabels(IMPACT_LEVELS)
    ax3.set_title('Impact rating per SFOP category (road-user view)', fontweight='bold')
    ax3.set_xlabel('Threat scenario'); ax3.set_ylabel('Impact category')

    matrix = np.array([[RISK_MATRIX[i][f] for f in FEASIBILITY_LEVELS] for i in IMPACT_LEVELS])
    ax4.imshow(matrix, cmap='viridis', vmin=1, vmax=5, origin='lower', alpha=0.6)
    for i in range(4):
        for j in range(4):
            ax4.text(j, i + 0.36, f'risk {matrix[i, j]}', ha='center', va='center', fontsize=8, color='0.25')
    offsets = {}
    for _, r in df.iterrows():
        x, y = FEASIBILITY_LEVELS.index(r.feasibility), IMPACT_LEVELS.index(r.impact)
        k = offsets.get((x, y), 0); offsets[(x, y)] = k + 1
        ax4.text(x, y + 0.12 - 0.17 * k, r.id, ha='center', va='center', fontsize=9, fontweight='bold')
    ax4.set_xticks(range(4)); ax4.set_xticklabels(FEASIBILITY_LEVELS)
    ax4.set_yticks(range(4)); ax4.set_yticklabels(IMPACT_LEVELS)
    ax4.set_xlabel('Attack feasibility rating'); ax4.set_ylabel('Impact rating')
    ax4.set_title('Example risk matrix (ISO/SAE 21434 Annex H)', fontweight='bold')
    plt.tight_layout(); plt.show()

    print('--- Figure (text summary) ---')
    print('  Risk value counts: ' + ', '.join(f'{k}: {v}' for k, v in risk_counts.items()))
    print('  CAL counts: ' + ', '.join(f'{k}: {v}' for k, v in zip(['none', 'CAL1', 'CAL2', 'CAL3', 'CAL4'], cal_counts.values)))

visualize_tara_results(df_tara)

## 3. Cybersecurity goals, claims and requirements

The risk treatment decision (15.9) leads to:
- a **cybersecurity goal** (clause 9.4) when the risk is *reduced*. A goal is a concept-level requirement that protects an asset's property against a threat scenario.
- a **cybersecurity claim** when the risk is *shared* or *retained*. A claim is a documented statement of why that is acceptable.

Goals are refined into the **cybersecurity concept** (9.5), i.e. cybersecurity requirements allocated to architecture elements, and later into technical specifications (clause 10).

**Match the control to the threat.** The original version of this notebook proposed "camera data shall be authenticated using cryptographic signatures" against an adversarial patch. This is a common mistake.
- A **physical adversarial input** (patch, dazzling, LiDAR spoofing) manipulates the *scene* in front of the sensor. The pixels arrive authentic and unmodified, so signatures cannot detect it. Suitable controls are diverse and redundant sensing with cross-checks, plausibility monitoring and robust or monitored ML. These causes must also be analysed under ISO 21448 (SOTIF) and ISO/PAS 8800.
- **Link integrity** protection (MAC/signature on the sensor-to-ECU or ECU-to-ECU link) addresses *tampering with data in transit*, e.g. a device spliced into the camera cable or forged bus frames.

> **Illustrative data:** goals and requirements below are examples for teaching, not a complete concept.

In [ ]:
# Concept-level content per threat (illustrative). The risk, CAL and decision columns are computed from the TARA above.
concept = {
    'T-001': ('CG-001: Protect the integrity of the object list used by AEB against physical-world adversarial inputs',
              ['CR-001.1: Pedestrian detections triggering AEB shall be confirmed by an independent sensor modality (radar)',
               'CR-001.2: The perception monitor shall flag implausible detection drop-outs of tracked objects',
               'CR-001.3: The camera-to-ECU link shall provide integrity/authenticity protection '
               '(addresses link tampering, NOT scene manipulation)']),
    'T-002': ('CG-002: Protect the integrity and authenticity of perception software and model weights',
              ['CR-002.1: Software and model packages shall be signed; signatures verified before installation (ISO 24089 process)',
               'CR-002.2: Secure boot shall verify the image at every start-up',
               'CR-002.3: Signing keys shall be stored in an HSM with access control and rollback protection']),
    'T-003': ('CG-003: Protect the integrity and authenticity of received V2X messages',
              ['CR-003.1: V2X messages shall be verified against IEEE 1609.2 / ETSI TS 103 097 certificates',
               'CR-003.2: Received V2X objects shall pass plausibility checks against on-board sensing before use',
               'CR-003.3: Misbehaviour shall be reported to the PKI for certificate revocation']),
    'T-004': ('CG-004: Protect the integrity of ECU firmware against tampering via diagnostics',
              ['CR-004.1: Reprogramming shall require UDS Authentication (0x29, certificate-based, ISO 14229-1:2020)',
               'CR-004.2: Diagnostic sessions and reprogramming attempts shall be logged']),
    'T-005': ('CG-005: Protect the availability of camera perception against dazzling (detect and degrade)',
              ['CR-005.1: Camera saturation/blinding shall be detected within the safety-relevant time budget',
               'CR-005.2: On detection, AEB shall enter a degraded mode with driver warning']),
    'T-006': ('CG-006: Protect the integrity of OTA packages in transit',
              ['CR-006.1: The OTA channel shall use mutually authenticated TLS 1.3',
               'CR-006.2: Package signatures shall be verified end-to-end, independent of the transport security']),
}

def treatment(risk):
    """Organisation-defined example policy: reduce risk values >= 2, retain risk value 1 with a claim."""
    return 'Reduce' if risk >= 2 else 'Retain'

rows = []
for _, r in df_tara.iterrows():
    goal, reqs = concept[r.id]
    decision = treatment(r.risk)
    for req in (reqs if decision == 'Reduce' else []):
        rows.append({'Threat': r.id, 'Risk': r.risk, 'Decision': decision,
                     'CAL': f'CAL {r.CAL}' if r.CAL else 'none', 'Goal': goal, 'Requirement': req})
df_requirements = pd.DataFrame(rows)
display(df_requirements)
print('--- Goals (text summary) ---')
for g, sub in df_requirements.groupby('Goal', sort=False):
    print(f"  {g} [{sub.CAL.iloc[0]}, risk {sub.Risk.iloc[0]}]: {len(sub)} requirements")

### Defence-in-depth layers

1. **Prevent** — authentication and authorisation, secure boot, input validation, segmentation.
2. **Detect** — intrusion detection (IDS), anomaly detection, logging. Notebook 4 builds a CAN IDS.
3. **Respond** — graceful degradation in the vehicle, incident response in the vehicle security operations centre (VSOC).
4. **Recover** — secure updates and recovery, post-incident analysis that feeds back into the TARA (clause 8).

## 4. Verification (clause 10) and validation (clause 11)

- **Verification** (cl. 10) checks that the specifications are implemented correctly. Methods include requirements-based tests, interface tests, static analysis, fuzzing and vulnerability scanning.
- **Validation** (cl. 11) checks at vehicle level that the cybersecurity goals are adequate and achieved. Penetration testing is a typical method.
- Vulnerabilities found are handled by **vulnerability analysis and management** (cl. 8.5, 8.6). Each one is either remediated or its risk is accepted with a documented rationale.

The simulation below tracks a penetration-test campaign. The release decision is based on the **open (unremediated)** vulnerabilities at the end of the campaign, not on the findings of the last week.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
def simulate_penetration_testing():
    """Synthetic penetration-test campaign with per-severity remediation tracking."""
    weeks = ['Wk 1-2', 'Wk 3-4', 'Wk 5-6', 'Wk 7-8', 'Wk 9-10', 'Wk 11-12']
    severities = ['Critical', 'High', 'Medium', 'Low']
    found = {'Critical': [3, 2, 1, 0, 0, 0], 'High': [5, 4, 3, 2, 1, 0],
             'Medium': [8, 7, 5, 4, 3, 2], 'Low': [10, 9, 8, 7, 6, 5]}
    fixed = {'Critical': [0, 3, 2, 1, 0, 0], 'High': [0, 3, 4, 3, 3, 1],
             'Medium': [0, 2, 5, 5, 4, 4], 'Low': [0, 0, 3, 4, 5, 6]}
    cum_found = {s: np.cumsum(found[s]) for s in severities}
    cum_fixed = {s: np.cumsum(fixed[s]) for s in severities}
    open_ = {s: cum_found[s] - cum_fixed[s] for s in severities}
    assert all((open_[s] >= 0).all() for s in severities)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
    x = np.arange(len(weeks)); bottom = np.zeros(len(weeks))
    colours = dict(zip(severities, [OKABE_ITO[5], OKABE_ITO[0], OKABE_ITO[1], OKABE_ITO[2]]))
    for s in severities:
        ax1.bar(x, found[s], 0.6, bottom=bottom, label=s, color=colours[s], edgecolor='black'); bottom += found[s]
    ax1.set_xticks(x); ax1.set_xticklabels(weeks)
    ax1.set_xlabel('Campaign period'); ax1.set_ylabel('Vulnerabilities found (count)')
    ax1.set_title('New findings per period', fontweight='bold'); ax1.legend()
    for s in severities:
        ax2.plot(x, open_[s], 'o-', lw=2, color=colours[s], label=f'{s} (open)')
    ax2.set_xticks(x); ax2.set_xticklabels(weeks)
    ax2.set_xlabel('Campaign period'); ax2.set_ylabel('Open vulnerabilities (count)')
    ax2.set_title('Open (unremediated) vulnerabilities', fontweight='bold'); ax2.legend()
    plt.tight_layout(); plt.show()

    total_found = sum(cum_found[s][-1] for s in severities)
    total_fixed = sum(cum_fixed[s][-1] for s in severities)
    print('Penetration-test campaign summary (synthetic)')
    for s in severities:
        print(f'  {s:8s}: found {cum_found[s][-1]:3d}, fixed {cum_fixed[s][-1]:3d}, open {open_[s][-1]:3d}')
    print(f'  Total found {total_found}, fixed {total_fixed}, open {total_found - total_fixed}')
    open_crit, open_high = open_['Critical'][-1], open_['High'][-1]
    if open_crit == 0 and open_high == 0:
        print('Release criterion met: no open critical/high findings. '
              'Remaining medium/low findings need a documented risk assessment (cl. 8.6).')
    else:
        print(f'Release criterion NOT met: {open_crit} critical and {open_high} high findings remain open.')
    return open_

open_vulns = simulate_penetration_testing()

## 5. Regulations and related standards (as of October 2026)

| Document | What it is | Relation to ISO/SAE 21434 |
|---|---|---|
| **UN Regulation No. 155** (2021) | Type-approval requirements for a cybersecurity management system (CSMS) and for vehicle-type cybersecurity | 21434 is the most widely used way to show the processes R155 requires. In the EU, R155 is mandatory for new vehicle types since July 2022 and for all newly registered vehicles since July 2024 ([UNECE](https://unece.org/transport/documents/2021/03/standards/un-regulation-no-155-cyber-security-and-cyber-security)) |
| **UN Regulation No. 156** (2021) | Software update management system (SUMS) and update requirements | Complements R155. The engineering counterpart is **ISO 24089:2023** (software update engineering) |
| **ISO/SAE PAS 5112:2022** | Guidelines for auditing cybersecurity engineering | Used to audit an organisation's 21434 processes (e.g. as input to a CSMS audit) |
| **ISO/SAE PAS 8475** | CAL and targeted attack feasibility (TAF) | Elaborates Annex E; in final approval stages, not yet listed as published |
| **EU Cyber Resilience Act**, Reg. (EU) 2024/2847 | Horizontal cybersecurity rules for products with digital elements | Art. 2(2)(c) excludes products in the scope of Reg. (EU) 2019/2144 (type-approved vehicles and their systems). Delegated Reg. (EU) 2025/1535 extends the exclusion to most L-category vehicles. **Aftermarket or non-type-approved connected parts remain in scope.** Reporting obligations apply from 11 September 2026; full application from 11 December 2027 ([EUR-Lex 2025/1535](https://eur-lex.europa.eu/eli/reg_del/2025/1535/oj/eng)) |
| SAE J3061 (2016) | Cybersecurity guidebook | Predecessor, superseded in practice by ISO/SAE 21434 |
| NHTSA, *Cybersecurity Best Practices for the Safety of Modern Vehicles* (2022) | US guidance (non-binding) | References ISO/SAE 21434 |

## 6. Relation to functional safety (ISO 26262)

Safety and security interact in both directions:

1. **Security → safety:** an attack can violate a safety goal (forged brake request → unintended braking).
2. **Safety → security:** safety features can be attack surfaces (diagnostic and reprogramming services).
3. **Security measures → safety:** cryptographic checks add latency and can drop messages. Their timing must fit the safety goal's FTTI.
4. **Common measures:** diverse redundancy helps against random faults *and* against single-sensor attacks. Its claimed effectiveness must be argued separately for each discipline.

The two standards have *analogous* work products, but they are **not equivalent**:

| ISO 26262 | ISO/SAE 21434 | Important difference |
|---|---|---|
| Item definition (Part 3, cl. 5) | Item definition (cl. 9.3) | Ideally one shared item definition |
| Hazard | Damage scenario | Damage scenarios also cover financial, operational and privacy harm |
| Hazardous event | Threat scenario | A threat scenario needs an asset, a cybersecurity property and a *cause* |
| HARA (S, E, C → ASIL) | TARA (impact × feasibility → risk value 1–5) | Feasibility of a *deliberate* attack replaces exposure and controllability |
| Safety goal | Cybersecurity goal / claim | Claims document retained or shared risk |
| ASIL (risk-derived integrity level) | CAL (informative, assurance rigour, from impact and attack vector) | A CAL is **not** derived from risk, and there is no ASIL↔CAL correspondence |
| Verification / validation (Parts 4, 6) | Verification (cl. 10) / validation (cl. 11) | Penetration testing is only one validation method |
| Field monitoring (Part 7) | Continual activities (cl. 8), incident response (cl. 13) | Security risk changes over time as attacker capabilities grow |

ISO 26262-2:2018 calls for communication channels between functional safety and cybersecurity, and gives examples of their interaction in its informative Annex E.
Notebook 2 works through this co-engineering for the AEB item.

In [ ]:
# Safety-security interaction examples (qualitative; no ASIL-CAL mapping is implied)
scenarios = pd.DataFrame([
    ('SS-001', 'Forged object list on the vehicle bus', 'Unintended emergency braking', 'Security -> safety', 'Bus message authentication (SecOC) + plausibility check in brake ECU'),
    ('SS-002', 'Tampered model weights via OTA', 'Missed pedestrian (fleet-wide)', 'Security -> safety', 'Signed updates, secure boot, post-update validation'),
    ('SS-003', 'Diagnostic reprogramming service', 'Firmware/calibration corruption', 'Safety feature -> attack surface', 'UDS 0x29 authentication, logging'),
    ('SS-004', 'MAC verification latency and frame drops', 'Late brake request', 'Security measure -> safety', 'Timing budget checked against FTTI; HW crypto'),
    ('SS-005', 'Camera dazzling', 'Perception unavailable', 'Common measure', 'Diverse sensing (camera + radar) and degradation strategy'),
], columns=['id', 'cause', 'safety-relevant effect', 'interaction', 'measure'])
display(scenarios)

counts = scenarios['interaction'].value_counts()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.barh(counts.index, counts.values, color=OKABE_ITO[4], edgecolor='black')
ax.set_xlabel('Number of example scenarios'); ax.set_ylabel('Interaction type')
ax.set_title('Safety-security interaction types in the examples', fontweight='bold')
ax.set_xticks(range(0, counts.max() + 1))
plt.tight_layout(); plt.show()
print('--- Interaction types (text summary) ---')
for k, v in counts.items():
    print(f'  {k}: {v}')

## Key takeaways

- ISO/SAE 21434:2021 is a **process** standard covering the full lifecycle (clauses 5–14) with TARA methods in clause 15. A second edition is in development (as of October 2026).
- TARA has seven steps, in order: assets & damage scenarios → threat scenarios → impact → attack paths → feasibility → risk value (1–5) → treatment decision.
- Attack potential **0–13 is High** feasibility; higher potential means a harder attack. There is no "Very High" class.
- Impact is rated for the **road user** using Severe/Major/Moderate/Negligible across Safety, Financial, Operational and Privacy.
- CAL (Annex E, informative) comes from impact and attack vector. It scales assurance rigour, and it is neither a risk value nor an ASIL equivalent.
- Controls must match the threat: link authentication does not stop physical-world adversarial inputs.
- UN R155/R156 make cybersecurity and update management a type-approval matter. The EU CRA excludes type-approved vehicles but not aftermarket connected parts.

## Self-assessment

1. Name the seven TARA steps of clause 15 in order.
2. An attack needs ≤1 week, a proficient attacker, public information, an unlimited window and standard equipment. What is the attack potential and the feasibility?
3. Why is a CAL not a risk measure? What two inputs does the Annex E example use?
4. Why can cryptographic signatures on camera frames not mitigate an adversarial patch?
5. Which clause covers vulnerability management, and which covers validation?
6. Is an aftermarket connected dashcam within the scope of the EU Cyber Resilience Act? Is a type-approved vehicle's telematics unit?

<details>
<summary>Answers</summary>

1. Asset identification and damage scenarios (15.3) → threat scenarios (15.4) → impact rating (15.5) → attack path analysis (15.6) → attack feasibility rating (15.7) → risk value determination (15.8) → risk treatment decision (15.9).
2. 1 + 3 + 0 + 0 + 0 = 4 → **High** feasibility.
3. A CAL expresses how much assurance rigour a cybersecurity goal needs. It does not express how likely or how bad an attack is. Annex E uses the **impact** and the **attack vector**, so a high-risk physical threat can get CAL 1 or 2.
4. The patch changes the scene, not the data. The camera faithfully captures and transmits the manipulated scene, so the frames are authentic. Signatures only detect tampering *after* capture.
5. Vulnerability analysis and management are in clause 8 (8.5, 8.6); cybersecurity validation is clause 11.
6. The dashcam is in scope as a product with digital elements, because it is not type-approved under Reg. (EU) 2019/2144. The type-approved vehicle's telematics unit is excluded by Art. 2(2)(c); UN R155 applies to it instead.
</details>

## References

- ISO/SAE 21434:2021, *Road vehicles — Cybersecurity engineering*. ISO/SAE, 2021. https://www.iso.org/standard/70918.html
- ISO 26262-2:2018, *Road vehicles — Functional safety — Part 2: Management of functional safety* (Annex E, interaction with cybersecurity).
- UNECE, UN Regulation No. 155 — Cyber security and cyber security management system (2021). https://unece.org/transport/documents/2021/03/standards/un-regulation-no-155-cyber-security-and-cyber-security
- UNECE, UN Regulation No. 156 — Software update and software update management system (2021). https://unece.org/transport/documents/2021/03/standards/un-regulation-no-156-software-update-and-software-update
- ISO/SAE PAS 5112:2022, *Road vehicles — Guidelines for auditing cybersecurity engineering*.
- ISO 24089:2023, *Road vehicles — Software update engineering*.
- Regulation (EU) 2024/2847 (Cyber Resilience Act). https://eur-lex.europa.eu/eli/reg/2024/2847/oj ; Delegated Regulation (EU) 2025/1535. https://eur-lex.europa.eu/eli/reg_del/2025/1535/oj/eng
- ISO/IEC 18045:2022, *Methodology for IT security evaluation* (origin of the attack-potential values).
- NHTSA (2022). *Cybersecurity Best Practices for the Safety of Modern Vehicles*. https://www.nhtsa.gov/document/cybersecurity-best-practices-safety-modern-vehicles-2022
- SAE J3061 (2016). *Cybersecurity Guidebook for Cyber-Physical Vehicle Systems* (superseded in practice by ISO/SAE 21434).
- Patel, M., Jung, R. (2026). Integrating Cybersecurity Into the AI Safety Assurance Argument: A GSN Pattern for AI-Based Perception Components. LNCS (Springer). https://doi.org/10.1007/978-3-032-35506-5_55

## Further practice

- Companion workshop exercise: [TARA exercise](https://github.com/milinpatel07/AI-Perception-Safety-Workshop/blob/main/notebooks/10_tara_exercise.ipynb) (AI Perception Safety Workshop).
- Template for your own analysis: [TARA template](../templates/TARA_Template.md).

---
**Navigation:** ← [Module 05 overview](README.md) · [Module index](README.md) · [Next: TARA](02_tara.ipynb) →